# skillspan-ner: train the final model and publish to Hugging Face

The winning sweep setup is **DeBERTa-v3 + a learned BIO CRF per head**: test F1 0.645 ± 0.005 over 3 seeds, validation 0.647.
This notebook retrains seed 43 (best on validation) and does four things:
1. pushes the checkpoint + CRF to the Hub
2. writes the model card using this run's real test numbers
3. reloads the model **from the Hub** as a smoke test
4. deploys the Gradio demo Space

**Settings:** Accelerator = GPU T4 (x2 is fine; only GPU 0 is used), Internet = On.
**Secret:** a Hugging Face *write* token, attached to this notebook as `hg`. The token is never printed.

In [ ]:
!git clone -q https://github.com/MSR327/skillspan-ner.git
%cd skillspan-ner
!pip install -q -U "transformers>=5.0" "accelerate>=1.1" datasets sentencepiece
!pip install -q -e ".[train]"
!git log --oneline -1

In [ ]:
import os

from huggingface_hub import HfApi
from kaggle_secrets import UserSecretsClient

os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("hg")  # child processes inherit it
api = HfApi()
me = api.whoami()
USER = me["name"]
role = me.get("auth", {}).get("accessToken", {}).get("role")
print("HF user:", USER, "| token role:", role)
assert role in (None, "write", "fineGrained"), "token must be a WRITE token"

MODEL_ID = f"{USER}/deberta-v3-skillspan-crf"
SPACE_ID = f"{USER}/skill-extractor"
print("model:", MODEL_ID, "\nspace:", SPACE_ID)

## Train (≈8 min) and push the best checkpoint

In [ ]:
!CUDA_VISIBLE_DEVICES=0 python -m skillex.train --config configs/deberta_crf.yaml --set hub_model_id={MODEL_ID}

## Model card with this run's numbers

In [ ]:
import json
from pathlib import Path

m = json.loads(Path("metrics/deberta-crf.json").read_text())
card = Path("MODEL_CARD.md").read_text()
fill = {
    "{{MODEL_ID}}": MODEL_ID,
    "{{SPACE_URL}}": f"https://huggingface.co/spaces/{SPACE_ID}",
    "{{TEST_F1}}": f'{m["test_crf_f1"]:.4f}',
    "{{TEST_F1_SKILL}}": f'{m["test_crf_f1_skill"]:.4f}',
    "{{TEST_F1_KNOWLEDGE}}": f'{m["test_crf_f1_knowledge"]:.4f}',
}
for k, v in fill.items():
    card = card.replace(k, v)
assert "{{" not in card
api.upload_file(path_or_fileobj=card.encode(), path_in_repo="README.md", repo_id=MODEL_ID,
                commit_message="Model card")
api.upload_file(path_or_fileobj="metrics/deberta-crf.json", path_in_repo="metrics.json",
                repo_id=MODEL_ID, commit_message="Test metrics, config and validation history")
print({k: v for k, v in fill.items() if "F1" in k})

## Smoke test: load from the Hub (not local files) and extract

In [ ]:
import shutil

from skillex.predict import SkillExtractor

shutil.rmtree("outputs", ignore_errors=True)  # prove nothing is loaded from disk
ex = SkillExtractor(MODEL_ID, device="cuda")
print("decoder:", ex.decode)
text = ("We are looking for a Data Engineer with strong Python and SQL experience. You will design ETL "
        "pipelines on AWS, collaborate with stakeholders, and mentor junior engineers.")
r = ex.extract(text)
print("skills:", r["skills"])
print("knowledge:", r["knowledge"])
assert ex.decode == "crf" and (r["skills"] or r["knowledge"])

## Deploy the Gradio Space

In [ ]:
api.create_repo(SPACE_ID, repo_type="space", space_sdk="gradio", exist_ok=True)
api.add_space_variable(SPACE_ID, "MODEL_ID", MODEL_ID)
api.upload_folder(repo_id=SPACE_ID, repo_type="space", folder_path="space",
                  ignore_patterns=["__pycache__/*", "*.pyc"], commit_message="Deploy demo")
print(f"model: https://huggingface.co/{MODEL_ID}")
print(f"space: https://huggingface.co/spaces/{SPACE_ID}  (first build takes a few minutes)")